# Laboratorio 3 - Reconocimiento de Lenguaje de Señas (ASL)
## CC3084 - Data Science, UVG

SignBridge es una startup guatemalteca que quiere construir un traductor de Lenguaje de Señas Americano (ASL) en tiempo real. En este laboratorio construimos y evaluamos un primer prototipo del motor de reconocimiento: un clasificador de letras del alfabeto ASL a partir de imágenes de manos, usando el dataset [ASL Alphabet de Kaggle](https://www.kaggle.com/datasets/grassknoted/asl-alphabet).

## Importando los paquetes necesarios

In [1]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.image import imread
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import load_img, img_to_array

## Cargando el conjunto de datos

In [ ]:
train_dir = "archive/asl_alphabet_train/asl_alphabet_train/"
test_dir = "archive/asl_alphabet_test/asl_alphabet_test/"

clases = sorted(os.listdir(train_dir))
print("Cantidad de clases:", len(clases))
print(clases)

## Ejercicio 1: Ejemplos de letras del alfabeto ASL

Mostramos 5 letras distintas con varias muestras cada una, para observar la variabilidad dentro de una misma clase.

In [ ]:
random.seed(42)
letras_ejemplo = ["A", "B", "M", "N", "S"]
muestras_por_letra = 5

fig, axes = plt.subplots(len(letras_ejemplo), muestras_por_letra, figsize=(12, 12))
for i, letra in enumerate(letras_ejemplo):
    archivos = os.listdir(train_dir + letra)
    seleccion = random.sample(archivos, muestras_por_letra)
    for j, archivo in enumerate(seleccion):
        imagen = imread(train_dir + letra + "/" + archivo)
        axes[i, j].imshow(imagen)
        axes[i, j].axis("off")
        if j == 0:
            axes[i, j].set_title(letra, fontsize=14, loc="left")

plt.suptitle("Ejemplos de letras y variabilidad dentro de cada clase")
plt.tight_layout()
plt.show()

Dentro de una misma letra las fotos varían en iluminación, posición de la mano dentro del cuadro y fondo, aunque todas fueron tomadas en un set up similar (misma persona, mismo fondo por sesión de captura).

## Ejercicio 2: Análisis exploratorio de los datos

### Resolución y formato de las imágenes

In [ ]:
resoluciones = set()
formatos = set()
modos = set()

for letra in clases:
    archivos = os.listdir(train_dir + letra)[:5]
    for archivo in archivos:
        img = Image.open(train_dir + letra + "/" + archivo)
        resoluciones.add(img.size)
        formatos.add(img.format)
        modos.add(img.mode)

print("Resoluciones encontradas:", resoluciones)
print("Formatos encontrados:", formatos)
print("Modos de color encontrados:", modos)

Todas las imágenes revisadas son archivos `.jpg` de 200x200 píxeles en color (RGB), consistente con lo que describe el dataset en Kaggle.

### Distribución de clases

In [ ]:
conteo_clases = {letra: len(os.listdir(train_dir + letra)) for letra in clases}
conteo_clases = pd.Series(conteo_clases).sort_index()

plt.figure(figsize=(12, 5))
conteo_clases.plot(kind="bar")
plt.title("Cantidad de imágenes por clase")
plt.ylabel("Cantidad de imágenes")
plt.xlabel("Clase")
plt.show()

conteo_clases.describe()

El dataset está balanceado: las 29 clases (A-Z, `space`, `del`, `nothing`) tienen 3,000 imágenes cada una, para un total de 87,000 imágenes de entrenamiento.

### Letras que se confunden visualmente

In [ ]:
grupos_confusos = [["M", "N", "S"], ["U", "V", "R"]]

for grupo in grupos_confusos:
    fig, axes = plt.subplots(1, len(grupo), figsize=(9, 3))
    for ax, letra in zip(axes, grupo):
        archivo = os.listdir(train_dir + letra)[0]
        imagen = imread(train_dir + letra + "/" + archivo)
        ax.imshow(imagen)
        ax.set_title(letra)
        ax.axis("off")
    plt.show()

**M/N/S**: las tres son puños cerrados, la diferencia está en la posición del pulgar (debajo de dos dedos, debajo de un dedo, o cruzado al frente), un detalle fino que un modelo puede pasar por alto fácilmente.

**U/V/R**: las tres muestran dos dedos extendidos, la diferencia está en si están juntos (U), separados en V, o cruzados (R). También es una diferencia sutil de la posición relativa de los dedos más que de la forma general de la mano.

Estas letras son las candidatas más probables a confusión en la matriz de confusión de los modelos.

### Definición del conjunto de entrenamiento/validación/prueba

El set de prueba oficial de Kaggle solo trae 1 imagen por clase (29 en total), insuficiente para evaluar el modelo. Por lo tanto construimos nuestro propio split a partir de las imágenes de entrenamiento.

Por el costo computacional de cargar 87,000 imágenes de 200x200 a color, tomamos una submuestra de 600 imágenes por clase (17,400 imágenes en total, ~20% del dataset completo), suficiente para entrenar y comparar modelos sin agotar la memoria disponible. Dividimos esa submuestra en 70% entrenamiento, 15% validación y 15% prueba, de forma estratificada para mantener el balance de clases en los tres conjuntos.

In [ ]:
random.seed(42)
muestras_por_clase = 600

rutas = []
etiquetas = []
for letra in clases:
    archivos = os.listdir(train_dir + letra)
    seleccion = random.sample(archivos, muestras_por_clase)
    for archivo in seleccion:
        rutas.append(train_dir + letra + "/" + archivo)
        etiquetas.append(letra)

datos = pd.DataFrame({"ruta": rutas, "etiqueta": etiquetas})
print(datos.shape)
datos.head()

In [ ]:
train_df, temp_df = train_test_split(datos, test_size=0.3, stratify=datos["etiqueta"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["etiqueta"], random_state=42)

print("Entrenamiento:", train_df.shape[0])
print("Validación:", val_df.shape[0])
print("Prueba:", test_df.shape[0])

## Ejercicio 3: Preprocesamiento de las imágenes

Antes de entrenar reducimos la resolución de 200x200 a 64x64 (menos parámetros y entrenamiento más rápido, sin perder la forma general de la mano) y normalizamos los píxeles a un rango de 0 a 1. No aplicamos filtros adicionales (por ejemplo escala de grises) porque el color puede ayudar al modelo a separar la mano del fondo.

In [ ]:
ejemplo_original = imread(train_dir + "A/" + os.listdir(train_dir + "A")[0])
ejemplo_reducido = img_to_array(load_img(train_dir + "A/" + os.listdir(train_dir + "A")[0], target_size=(64, 64)))

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(ejemplo_original)
axes[0].set_title(f"Original {ejemplo_original.shape[:2]}")
axes[0].axis("off")
axes[1].imshow(ejemplo_reducido.astype("uint8"))
axes[1].set_title(f"Reducida {ejemplo_reducido.shape[:2]}")
axes[1].axis("off")
plt.show()

In [ ]:
tamano_imagen = 64

def cargar_y_procesar(df):
    imagenes = []
    for ruta in df["ruta"]:
        imagen = load_img(ruta, target_size=(tamano_imagen, tamano_imagen))
        imagenes.append(img_to_array(imagen))
    return np.asarray(imagenes) / 255.0

X_train = cargar_y_procesar(train_df)
X_val = cargar_y_procesar(val_df)
X_test = cargar_y_procesar(test_df)

print(X_train.shape, X_val.shape, X_test.shape)
print("Rango de valores:", X_train.min(), "-", X_train.max())

In [ ]:
etiqueta_a_indice = {letra: i for i, letra in enumerate(clases)}

y_train = train_df["etiqueta"].map(etiqueta_a_indice).values
y_val = val_df["etiqueta"].map(etiqueta_a_indice).values
y_test = test_df["etiqueta"].map(etiqueta_a_indice).values

print(y_train.shape, y_val.shape, y_test.shape)